# Notebook 01 — Diagnostic des variantes rares

> **Objectif** : comprendre *pourquoi* les variantes avec < 6 observations posent un problème spécifique, le quantifier, et mettre en place deux métriques de détection de l'over-fitting.

---

## Concepts abordés
1. Le biais-variance trade-off — intuition géométrique
2. Simulation de l'over-fitting sur données rares
3. Métrique 1 : ratio LOO-CV RMSE
4. Métrique 2 : coefficient de variation des prédictions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from pathlib import Path

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_parquet(Path('data/dataset_industriel.parquet'))
FEATURES = ['composants', 'mo', 'energie', 'volume']
TARGET   = 'cout'
print(f"Dataset chargé : {len(df):,} lignes")

## 1. Intuition : le biais-variance trade-off

L'erreur de généralisation d'un modèle se décompose en trois termes :

$$\text{MSE} = \underbrace{\text{Biais}^2}_{\text{erreur systématique}} + \underbrace{\text{Variance}}_{\text{sensibilité aux données}} + \underbrace{\sigma^2_{\varepsilon}}_{\text{bruit irréductible}}$$

**Avec peu de données**, un modèle flexible (biais faible) a une variance explosive.  
**Solution** : introduire volontairement du biais (régularisation, prior) pour réduire la variance.

In [ ]:
# --- Simulation pédagogique du biais-variance ---
# On simule la vraie fonction f(x) = sin(x) + bruit
# et on montre comment un modèle polynomial se comporte avec peu de données

def true_function(x):
    return 2 * np.sin(x) + 0.5 * x

x_grid = np.linspace(0, 2 * np.pi, 200)
y_true = true_function(x_grid)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, n_points, title in zip(
        axes,
        [3, 7, 30],
        ['3 observations\n(variante rare)', '7 observations\n(zone grise)', '30 observations\n(variante mature)']):

    ax.plot(x_grid, y_true, 'k--', lw=2, label='Vraie fonction')

    for trial in range(15):  # 15 jeux de données différents
        x_sample = np.random.uniform(0, 2 * np.pi, n_points)
        y_sample = true_function(x_sample) + np.random.normal(0, 0.5, n_points)

        # Fit polynomial degré 3
        coeffs = np.polyfit(x_sample, y_sample, deg=min(3, n_points - 1))
        y_pred = np.polyval(coeffs, x_grid)
        ax.plot(x_grid, y_pred, alpha=0.3, color='#e74c3c', lw=1)

    ax.set_xlim(0, 2 * np.pi)
    ax.set_ylim(-5, 10)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('x')

axes[0].set_ylabel('Coût prédit')
axes[0].legend(loc='upper left')
fig.suptitle('Variance du modèle selon le nombre d\'observations\n'
             '(chaque courbe rouge = un jeu de données différent)', fontsize=13)
plt.tight_layout()
plt.savefig('data/fig_bias_variance.png', dpi=150)
plt.show()

print("Observation clé : avec 3 points, les courbes rouges divergent massivement\n"
      "→ le modèle est très sensible au jeu de données particulier (haute variance)")

## 2. Over-fitting sur variantes rares : simulation réaliste

On ajuste une régression linéaire sur chaque variante individuellement et on compare le RMSE in-sample entre variantes rares et matures.

In [ ]:
def fit_individual_model(sub_df: pd.DataFrame) -> dict:
    """Ajuste OLS sur les données d'une variante. Retourne les métriques."""
    X = sub_df[FEATURES].values
    y = sub_df[TARGET].values
    n = len(y)

    if n < 2:
        return {'n_obs': n, 'rmse_insample': np.nan, 'r2': np.nan}

    model = LinearRegression()
    model.fit(X, y)
    y_hat = model.predict(X)

    rmse = np.sqrt(mean_squared_error(y, y_hat))
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return {'n_obs': n, 'rmse_insample': rmse, 'r2': r2}


results_individual = (
    df.groupby('variante')
      .apply(fit_individual_model)
      .apply(pd.Series)
      .reset_index()
)
results_individual['categorie'] = pd.cut(
    results_individual['n_obs'], bins=[0, 5, 24, 200],
    labels=['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']
)

print(results_individual.groupby('categorie')[['rmse_insample', 'r2']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {'Rare (≤5)': '#e74c3c', 'Interméd. (6-24)': '#f39c12', 'Mature (≥25)': '#27ae60'}

ax = axes[0]
for cat, color in palette.items():
    data = results_individual[results_individual['categorie'] == cat]['rmse_insample'].dropna()
    ax.hist(data, bins=20, alpha=0.6, color=color, label=cat)
ax.set_xlabel('RMSE in-sample (€)')
ax.set_ylabel('Nombre de variantes')
ax.set_title('RMSE in-sample par catégorie')
ax.legend()

ax = axes[1]
for cat, color in palette.items():
    data = results_individual[results_individual['categorie'] == cat]['r2'].dropna()
    ax.hist(data, bins=20, alpha=0.6, color=color, label=cat)
ax.set_xlabel('R² in-sample')
ax.set_title('R² in-sample par catégorie\n(R²→1 sur les rares = signe d\'over-fit !)')
ax.legend()

plt.tight_layout()
plt.savefig('data/fig_overfitting_insample.png', dpi=150)
plt.show()
print("\nNote : un R² proche de 1 sur les variantes rares ne signifie PAS\n"
      "que le modèle est bon — il signifie qu'il a mémorisé le bruit !")

## 3. Métrique 1 : ratio LOO-CV RMSE

**LOO-CV (Leave-One-Out Cross-Validation)** : on entraîne le modèle sur n-1 points et on prédit le point laissé de côté. On répète pour chaque point.

$$\text{LOO-RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_{-i})^2}$$

**Ratio LOO-CV** = LOO-RMSE / RMSE in-sample
- Ratio ≈ 1.0 → 1.5 : modèle sain
- Ratio > 2.5 : **over-fitting détecté**

In [ ]:
def loo_cv_rmse(sub_df: pd.DataFrame) -> float:
    """Calcule le RMSE LOO-CV pour une variante donnée."""
    X = sub_df[FEATURES].values
    y = sub_df[TARGET].values
    n = len(y)

    if n < 3:
        return np.nan  # LOO non calculable avec < 3 points

    residuals = []
    for i in range(n):
        # Train sans l'observation i
        X_train = np.delete(X, i, axis=0)
        y_train = np.delete(y, i)
        X_test  = X[i:i+1]
        y_test  = y[i]

        model = LinearRegression()
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)[0]
            residuals.append((y_test - y_pred) ** 2)
        except Exception:
            residuals.append(np.nan)

    return np.sqrt(np.nanmean(residuals))


print("Calcul du LOO-CV (peut prendre 30s)...")
loo_results = (
    df.groupby('variante')
      .apply(lambda g: pd.Series({
          'n_obs'       : len(g),
          'rmse_insample': np.sqrt(mean_squared_error(
              g[TARGET], LinearRegression().fit(g[FEATURES], g[TARGET]).predict(g[FEATURES])
          )) if len(g) >= 2 else np.nan,
          'rmse_loo'    : loo_cv_rmse(g),
      }))
      .reset_index()
)

loo_results['ratio_loo'] = loo_results['rmse_loo'] / loo_results['rmse_insample']
loo_results['categorie'] = pd.cut(
    loo_results['n_obs'], bins=[0, 5, 24, 200],
    labels=['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']
)

print("\nRatio LOO / in-sample par catégorie :")
print(loo_results.groupby('categorie')['ratio_loo'].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

palette = {'Rare (≤5)': '#e74c3c', 'Interméd. (6-24)': '#f39c12', 'Mature (≥25)': '#27ae60'}
for cat, color in palette.items():
    data = loo_results[loo_results['categorie'] == cat]['ratio_loo'].dropna()
    ax.scatter(loo_results.loc[loo_results['categorie'] == cat, 'n_obs'],
               data, color=color, alpha=0.7, label=cat, s=60)

ax.axhline(2.5, color='red', linestyle='--', lw=2, label='Seuil over-fit (ratio=2.5)')
ax.axhline(1.0, color='gray', linestyle=':', lw=1)
ax.set_xlabel('Nombre d\'observations')
ax.set_ylabel('Ratio LOO-RMSE / RMSE in-sample')
ax.set_title('Détection de l\'over-fitting : ratio LOO-CV')
ax.legend()
ax.set_ylim(0, 10)

plt.tight_layout()
plt.savefig('data/fig_loo_ratio.png', dpi=150)
plt.show()

n_overfit = (loo_results['ratio_loo'] > 2.5).sum()
print(f"\n{n_overfit} variantes avec ratio > 2.5 (over-fitting détecté)")

## 4. Métrique 2 : Coefficient de Variation des prédictions

**Idée** : au sein d'une même famille, les variantes rares ne devraient pas avoir une dispersion de coûts prédit très différente des variantes matures (sauf raison technologique).

Si les variantes rares ont un **CV prédit 3× supérieur** à celui des variantes matures de la même famille → c'est du bruit, pas du signal.

In [ ]:
# Calcul du coût moyen prédit par variante (moyenne des obs)
mean_pred_by_variant = df.groupby(['variante', 'famille', 'n_obs_variante'])[TARGET].mean().reset_index()
mean_pred_by_variant.rename(columns={TARGET: 'cout_moyen'}, inplace=True)
mean_pred_by_variant['categorie'] = pd.cut(
    mean_pred_by_variant['n_obs_variante'], bins=[0, 5, 24, 200],
    labels=['Rare', 'Interméd.', 'Mature']
)

# CV par famille et catégorie
def coeff_variation(x):
    return x.std() / x.mean() if x.mean() > 0 else np.nan

cv_by_family = (
    mean_pred_by_variant.groupby(['famille', 'categorie'])['cout_moyen']
    .apply(coeff_variation)
    .reset_index(name='cv')
)

cv_pivot = cv_by_family.pivot(index='famille', columns='categorie', values='cv')
cv_pivot['ratio_rare_mature'] = cv_pivot['Rare'] / cv_pivot['Mature']

print("CV des coûts par famille (rare vs mature) :")
print(cv_pivot.dropna().sort_values('ratio_rare_mature', ascending=False).round(3).head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

valid = cv_pivot.dropna(subset=['ratio_rare_mature'])
ax.bar(range(len(valid)), valid['ratio_rare_mature'].values,
       color=['#e74c3c' if r > 3 else '#27ae60' for r in valid['ratio_rare_mature'].values])
ax.axhline(3.0, color='red', linestyle='--', lw=2, label='Seuil alerte (ratio=3)')
ax.axhline(1.0, color='gray', linestyle=':', lw=1, label='Ratio idéal = 1')
ax.set_xticks(range(len(valid)))
ax.set_xticklabels(valid.index, rotation=45, ha='right')
ax.set_ylabel('CV(rare) / CV(mature)')
ax.set_title('Ratio de dispersion : variantes rares vs matures par famille')
ax.legend()

plt.tight_layout()
plt.savefig('data/fig_cv_ratio.png', dpi=150)
plt.show()

## Résumé du Notebook 01

| Concept | Ce qu'on a appris |
|---------|-------------------|
| Biais-variance | Avec peu de données, la variance du modèle explose — le même algo donne des résultats très différents selon le jeu de données |
| RMSE in-sample | Trompeuse : R²=1 sur variante rare = over-fit, pas performance réelle |
| Ratio LOO-CV | **Métrique 1** — ratio > 2.5 = signal clair d'over-fitting |
| CV prédit | **Métrique 2** — ratio > 3 entre variantes rares et matures = bruit |

**→ Notebook suivant : [02_mixed_effects_model.ipynb](02_mixed_effects_model.ipynb)**